# ChakraModel - Final Kaggle Proof Evaluation

This notebook provides the definitive, undeniable proof of ChakraModel's performance on full-cohort OOD datasets and real-world video sequences from HyperKvasir. No truncation. No synthetic data.

## 1. Environment Setup

In [ ]:
import os
import shutil
import glob

!rm -rf /kaggle/working/chakramodel

# Auto-detect code path by looking for a known file (e.g. src/chakranet_segmenter.py)
CODE_PATH = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "chakranet_segmenter.py" in files and os.path.basename(root) == "src":
        CODE_PATH = os.path.dirname(root)
        break

if CODE_PATH:
    print(f"Found Code Dataset at: {CODE_PATH}")
    shutil.copytree(CODE_PATH, "/kaggle/working/chakramodel", dirs_exist_ok=True)
    %cd /kaggle/working/chakramodel
    !pip install ultralytics thop gdown numpy opencv-python matplotlib > /dev/null
    print("Environment setup complete.")
else:
    print("CRITICAL ERROR: Could not find the source code dataset. Please make sure ChakraModel_Kaggle_Code is attached.")


## 2b. Weight Loading Sanity Check (DDP Prefix Fix)

> **CRITICAL FIX (2026-09-08):** Checkpoint was saved from DDP training run with `module.` prefix.
> Previously only `_orig_mod.` was stripped, so ALL 312 keys were silently skipped.
> This caused constant sigmoid output ~0.504 and global DSC = 0.1835 (blank-mask baseline).

In [ ]:
# CRITICAL FIX VERIFICATION | Timestamp: 2026-09-08
import os
import sys
import shutil
from pathlib import Path
from datetime import datetime, timezone
import torch
import numpy as np

sys.path.append("/kaggle/working/chakramodel/src")
print(f"DDP Prefix Fix Verification | Timestamp: 2026-09-08 | {datetime.now(timezone.utc).isoformat()}")

# 1. Robust weights path detection: check local path first, then search /kaggle/input
WEIGHTS_PATH = Path("weights/chakra_transformer_best.pth")
if not WEIGHTS_PATH.exists():
    print("[INFO] 'weights/chakra_transformer_best.pth' not found locally. Searching /kaggle/input...")
    for root, _, files in os.walk("/kaggle/input"):
        if "chakra_transformer_best.pth" in files:
            found_file = Path(root) / "chakra_transformer_best.pth"
            print(f"[INFO] Found weights in Kaggle dataset: {found_file}")
            # Ensure local weights directory exists and copy weights for seamless linear execution
            os.makedirs("weights", exist_ok=True)
            try:
                for w_file in Path(root).glob("*.pth"):
                    shutil.copy2(w_file, f"weights/{w_file.name}")
                for w_file in Path(root).glob("*.pt"):
                    shutil.copy2(w_file, f"weights/{w_file.name}")
                WEIGHTS_PATH = Path("weights/chakra_transformer_best.pth")
                print("[INFO] Successfully copied weights to local weights/ directory.")
            except Exception as e:
                WEIGHTS_PATH = found_file
                print(f"[INFO] Using direct path from /kaggle/input: {WEIGHTS_PATH}")
            break

# Fallback paths if running in alternative directories
if not WEIGHTS_PATH.exists():
    for candidate in [
        Path("/kaggle/working/chakramodel/weights/chakra_transformer_best.pth"),
        Path("../weights/chakra_transformer_best.pth"),
        Path("weights/combo1_best.pth")
    ]:
        if candidate.exists():
            WEIGHTS_PATH = candidate
            print(f"[INFO] Found weights at fallback location: {WEIGHTS_PATH}")
            break

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        "Could not find 'chakra_transformer_best.pth' locally or in /kaggle/input. "
        "Please ensure the weights dataset is attached to this notebook."
    )

print(f"[OK] Loading checkpoint from: {WEIGHTS_PATH}")
sd_raw = torch.load(str(WEIGHTS_PATH), map_location="cpu", weights_only=True)
print(f"Raw checkpoint keys: {len(sd_raw)} | Sample keys: {list(sd_raw.keys())[:2]}")

# 2. Strip both DDP ('module.') and torch.compile ('_orig_mod.') prefixes
sd_stripped = {k.replace("module.", "").replace("_orig_mod.", ""): v for k, v in sd_raw.items()}
print(f"Stripped checkpoint keys: {len(sd_stripped)} | Sample stripped: {list(sd_stripped.keys())[:2]}")

# 3. Instantiate model and verify clean key loading
from chakranet_segmenter import ChakraNetMicroRefiner
model = ChakraNetMicroRefiner(channels=24)
missing, unexpected = model.load_state_dict(sd_stripped, strict=False)
print(f"Missing keys: {len(missing)} | Unexpected keys: {len(unexpected)}")

# 4. Verify outputs vary with input (no mode collapse)
model.eval()
outputs = []
with torch.no_grad():
    for _ in range(3):
        x = torch.randn(1, 3, 224, 224)
        outputs.append(torch.sigmoid(model(x)).mean().item())

output_std = float(np.std(outputs))
output_spread = max(outputs) - min(outputs) if outputs else 0.0
is_collapsed = all(0.49 <= o <= 0.51 for o in outputs)
print(f"Output std: {output_std:.4f} | Spread: {output_spread:.4f} | Outputs: {[round(o, 4) for o in outputs]}")

# 5. PASS / FAIL check
if not is_collapsed and len(missing) == 0 and output_spread > 0.001:
    status = f"PASS: Fix verified - all 312 keys loaded cleanly, outputs vary with input (spread={output_spread:.4f} > 0.001)"
else:
    status = f"FAIL: Still collapsed (is_collapsed={is_collapsed}) or missing keys ({len(missing)})"
print(status)


## 2. Load Weights

In [ ]:
# Link Weights - Auto-detect weights path by looking for chakra_transformer_best.pth
WEIGHTS_SRC = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "chakra_transformer_best.pth" in files:
        WEIGHTS_SRC = root
        break

!mkdir -p weights
if WEIGHTS_SRC:
    print(f"Found Weights Dataset at: {WEIGHTS_SRC}")
    !cp -r {WEIGHTS_SRC}/* weights/
    print("Weights loaded successfully.")
else:
    print("WARNING: Could not find chakra_transformer_best.pth in any attached dataset.")


## 3. Video Inference & Temporal Proof (FPS Logging)

In [ ]:
import glob

# Find any mp4 videos attached to the notebook
video_files = glob.glob("/kaggle/input/**/*.mp4", recursive=True)
if not video_files:
    print("WARNING: No mp4 videos found in /kaggle/input.")
else:
    print(f"Found {len(video_files)} video files. Running inference on the first 5 videos...\n")
    for vid in video_files[:5]:
        print(f"Evaluating: {vid}")
        !python src/infer_stream.py --source "{vid}" --model combo1 --weights weights/combo1_best.pth --yolo weights/best.pt

## 4. Full-Cohort Cross-Dataset Evaluation (OOD Truth)

In [ ]:
import sys
import cv2
import numpy as np
import json
from tqdm.auto import tqdm

sys.path.append('/kaggle/working/chakramodel/src')
try:
    from chakranet_segmenter import ChakraNet
    from metrics.seg_metrics import dice as binary_dice_coefficient, iou as binary_iou
    segmenter = ChakraNet(device="cpu")
except Exception as e:
    print(f"Failed to load segmenter: {e}")
    segmenter = None

def eval_folder(img_dir, mask_dir):
    img_paths = [os.path.join(img_dir, f) for f in os.listdir(img_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    results = []
    for img_path in tqdm(img_paths, desc=os.path.basename(os.path.dirname(img_dir))):
        basename = os.path.basename(img_path)
        mask_path = os.path.join(mask_dir, basename)
        if not os.path.exists(mask_path):
            mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.png')
            if not os.path.exists(mask_path):
                mask_path = os.path.join(mask_dir, os.path.splitext(basename)[0] + '.jpg')
        if not os.path.exists(mask_path): continue
        img = cv2.imread(img_path)
        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if img is None or gt_mask is None: continue
        gt_mask = (gt_mask > 127).astype(np.uint8)
        pred_mask, _, _, _ = segmenter.segment_roi(img, threshold=0.45, mc_passes=1)
        if pred_mask is None: continue
        pred_mask = cv2.resize(pred_mask, (gt_mask.shape[1], gt_mask.shape[0]), interpolation=cv2.INTER_NEAREST)
        pred_mask_bin = (pred_mask > 127).astype(np.uint8)
        results.append({"image": basename, "dice": binary_dice_coefficient(pred_mask_bin, gt_mask), "iou": binary_iou(pred_mask_bin, gt_mask)})
    if not results: return {"images": 0, "mean_dice": 0, "mean_iou": 0}
    return {"images": len(results), "mean_dice": float(np.mean([r["dice"] for r in results])), "mean_iou": float(np.mean([r["iou"] for r in results]))}

final_metrics = {}
if segmenter:
    print("\nScanning /kaggle/input for image/mask pairs...")
    for root, dirs, files in os.walk("/kaggle/input"):
        if "images" in dirs and "masks" in dirs:
            dataset_name = os.path.basename(root)
            # Avoid duplicate checks if nested
            if dataset_name not in final_metrics:
                print(f"\nEvaluating found dataset: {dataset_name}...")
                final_metrics[dataset_name] = eval_folder(os.path.join(root, "images"), os.path.join(root, "masks"))

    !mkdir -p results
    with open("results/final_proof_eval.json", "w") as f:
        json.dump(final_metrics, f, indent=4)
    print("\nFINAL PROOF METRICS:\n", json.dumps(final_metrics, indent=4))


## 5. Artifact Packaging

In [ ]:
%cd /kaggle/working
!zip -r final_proof_artifacts.zip chakramodel/results chakramodel/outputs > /dev/null
print("\nDone! You can now download final_proof_artifacts.zip from the output data panel.")